# 📈 Real-Time Bitcoin Price Analysis on SageMaker
In this notebook, we will build a real-time Bitcoin price ingestion pipeline on AWS using:
- CoinGecko API → to get live Bitcoin price data.
- Amazon Kinesis Data Streams → to ingest and stream data.
- Amazon S3 → to store data.
- SageMaker → to later process, train, and deploy ML models (future steps).

This section covers:
AWs setup  
S3 bucket creation  
Kinesis Data Stream setup  
Fetching Bitcoin prices from CoinGecko  
Sending data to Kinesis


In [ ]:
import boto3
import json
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

region = "us-east-2
session = boto3.Session(region_name=region)
s3_client = session.client('s3')
kinesis_client = session.client('kinesis')
sagemaker_client = session.client('sagemaker')
import sagemaker
sagemaker_session = sagemaker.Session(boto_session=session)

role = "<arn:aws:iam::491992923505:role/sagemakerRole>"

print("AWS clients initialized for region:", region)

Raw bitcoin prices will be stored below

In [ ]:
#a unique S3 bucket for this project
bucket_name = f"bitcoin-price-data-{int(time.time())}"
try:
    if region == "us-east-1":
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})
    print(f"Created S3 bucket: {bucket_name}")
except Exception as e:
    print(f"Bucket creation failed (bucket may already exist): {e}")


we use kinesis to streaam bitcoin prices

In [ ]:
# a Kinesis Data Stream for streaming the price data
stream_name = "BitcoinPriceStream"
try:
    kinesis_client.create_stream(StreamName=stream_name, ShardCount=1)
    waiter = kinesis_client.get_waiter('stream_exists')
    waiter.wait(StreamName=stream_name)
    stream_desc = kinesis_client.describe_stream(StreamName=stream_name)
    status = stream_desc['StreamDescription']['StreamStatus']
    print(f"Kinesis stream '{stream_name}' created. Status: {status}")
except Exception as e:
    print(f"Stream creation failed (maybe already exists): {e}")

60 days of hourly bitcoin data will be pushed

In [ ]:

coin_id = "bitcoin"
vs_currency = "usd"
days = 60
url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart?vs_currency={vs_currency}&days={days}"
response = requests.get(url)
data = response.json()

# The API returns a dict with 'prices', 'market_caps', 'total_volumes'. We'll use 'prices'.
prices = data.get("prices", [])
print(f"Retrieved {len(prices)} price points from CoinGecko for the last {days} days.")
# Example of first record
if prices:
    ts_example, price_example = prices[0]
    print("Example data point:", datetime.utcfromtimestamp(ts_example/1000).strftime('%Y-%m-%d %H:%M:%S'), price_example)



In [ ]:
# Send the data points to Kinesis stream as if ingesting in real-time
partition_key = "btc-price"
for ts, price in prices:
    timestamp_str = datetime.utcfromtimestamp(ts/1000).strftime('%Y-%m-%dT%H:%M:%SZ')
    record = f"{timestamp_str},{price}"
    kinesis_client.put_record(
        StreamName=stream_name,
        Data=record.encode('utf-8'),
        PartitionKey=partition_key
    )
print(f"Streamed {len(prices)} records into Kinesis stream '{stream_name}'.")

In future steps, we will build on this ingestion pipeline to perform real-time Bitcoin price analysis using SageMaker. We will process the data from Kinesis, store it in S3, and train a time series forecasting model (DeepAR) to predict future Bitcoin prices. 